In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

## Plotting Functionality 

In [ ]:
def load_runs(output_dir, env_id, alg_name, final=False):
    """
    Discover and load all returns.npy files for one algorithm/environment.
    Returns a list of arrays, each shape (N_i, 2): columns [global_step, episodic_return].
    Raises FileNotFoundError if no runs are found.
    """
    if final:
        pattern = os.path.join(output_dir, env_id, alg_name, "final_run", "*", "returns.npy")
    else:
        pattern = os.path.join(output_dir, env_id, alg_name, "*", "returns.npy")
    paths = sorted(glob.glob(pattern))
    print(paths)
    if not paths:
        raise FileNotFoundError(f"No runs found matching: {pattern}")
    print(f"  {alg_name}: found {len(paths)} run(s)")
    return [np.load(p) for p in paths]


def _ema_smooth(values, weight):
    """Exponential moving average. weight=0 means no smoothing."""
    if weight == 0:
        return values
    smoothed, last = [], values[0]
    for v in values:
        last = weight * last + (1 - weight) * v
        smoothed.append(last)
    return np.array(smoothed)


def plot_runs(
    output_dir,
    env_id,
    alg_name,
    label=None,
    color=None,
    smoothing=0.0,
    num_points=300,
    ax=None,
    final=False
):
    """
    Load all seeds for one algorithm and plot mean ± 1 std learning curve.

    Parameters
    ----------
    output_dir : str   Root output directory (e.g. "output" or "../output")
    env_id     : str   Gymnasium environment id (e.g. "HalfCheetah-v4")
    alg_name   : str   Algorithm subdirectory name ("ppo" or "sac")
    label      : str   Legend label; defaults to alg_name.upper()
    color      : str   Matplotlib color string; auto-assigned if None
    smoothing  : float EMA weight in [0, 1). 0 = no smoothing, 0.9 = heavy smoothing
    num_points : int   Number of evenly-spaced x-axis points for interpolation
    ax         : matplotlib Axes object; creates a new figure if None

    Returns
    -------
    ax : the Axes object (for chaining / further customisation)
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))
    if label is None:
        label = alg_name.upper()

    runs = load_runs(output_dir, env_id, alg_name, final)

    # Determine common x-axis: from the latest first-step to the earliest last-step
    x_min = max(r[0, 0]  for r in runs)
    x_max = min(r[-1, 0] for r in runs)
    if x_min >= x_max:
        raise ValueError(
            f"Runs for {alg_name} have no overlapping step range. "
            "Check that all seeds completed at least one episode."
        )
    steps = np.linspace(x_min, x_max, num_points)

    # Smooth each seed, then interpolate onto the common axis
    interp = np.stack([
        np.interp(steps, r[:, 0], _ema_smooth(r[:, 1], smoothing))
        for r in runs
    ])  # shape: (num_seeds, num_points)

    mean = interp.mean(axis=0)
    std  = interp.std(axis=0)

    line, = ax.plot(steps, mean, label=label, color=color, linewidth=1.8)
    c = line.get_color()
    ax.fill_between(steps, mean - std, mean + std, color=c, alpha=0.2)

    return ax

## Plotting PPO 

In [ ]:
# ── Edit these to match your runs ─────────────────────────────────────────────
OUTPUT_DIR = "../agents/output"   # path to the output/ folder relative to this notebook
ENV_ID     = "Pusher-v4" # Change to your chosen environments
SMOOTHING  = 0.9           # EMA weight: 0 = raw data, 0.9 = smooth
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# Useful for quick inspection of one algorithm
fig, ax = plt.subplots(figsize=(8, 5))
plot_runs(OUTPUT_DIR, ENV_ID, "ppo", smoothing=SMOOTHING, ax=ax)
ax.set_xlabel("Environment Steps"); ax.set_ylabel("Episodic Return")
ax.set_title(f"PPO — {ENV_ID}"); ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout(); plt.show()

## Plotting SAC

In [ ]:
# Useful for quick inspection of one algorithm
fig, ax = plt.subplots(figsize=(8, 5))
plot_runs(OUTPUT_DIR, ENV_ID, "sac", smoothing=SMOOTHING, ax=ax)
ax.set_xlabel("Environment Steps"); ax.set_ylabel("Episodic Return")
ax.set_title(f"PPO — {ENV_ID}"); ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout(); plt.show()

## Plotting PPO vs SAC

In [ ]:
# ── Edit these to match your runs ─────────────────────────────────────────────
OUTPUT_DIR = "../agents/output"   # path to the output/ folder relative to this notebook
ENV_ID     = "Pusher-v4" # Change to your chosen environments
SMOOTHING  = 0.9           # EMA weight: 0 = raw data, 0.9 = smooth
# ──────────────────────────────────────────────────────────────────────────────v

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_runs(OUTPUT_DIR, ENV_ID, "ppo", label="PPO", ax=ax, smoothing=SMOOTHING)
plot_runs(OUTPUT_DIR, ENV_ID, "sac", label="SAC", ax=ax, smoothing=SMOOTHING)

ax.set_xlabel("Environment Steps", fontsize=12)
ax.set_ylabel("Episodic Return",   fontsize=12)
ax.set_title(f"PPO vs SAC — {ENV_ID}", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.4)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

plt.tight_layout()
plt.savefig(f"{ENV_ID}_learning_curves.pdf", bbox_inches="tight")
plt.show()

## Plotting Final PPO vs SAC 

In [ ]:
# ── Edit these to match your runs ─────────────────────────────────────────────
OUTPUT_DIR = "../agents/output/"   # path to the output/ folder relative to this notebook
ENV_ID     = "Pusher-v4" # Change to your chosen environments
SMOOTHING  = 0.9           # EMA weight: 0 = raw data, 0.9 = smooth
# ──────────────────────────────────────────────────────────────────────────────v

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_runs(OUTPUT_DIR, ENV_ID, "ppo", label="PPO", ax=ax, smoothing=SMOOTHING, final=True)
plot_runs(OUTPUT_DIR, ENV_ID, "sac", label="SAC", ax=ax, smoothing=SMOOTHING, final=True)

ax.set_xlabel("Environment Steps", fontsize=12)
ax.set_ylabel("Episodic Return",   fontsize=12)
ax.set_title(f"PPO vs SAC — {ENV_ID}", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.4)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

plt.tight_layout()
plt.savefig(f"{ENV_ID}_learning_curves.pdf", bbox_inches="tight")
plt.show()